# Download a PDF from Deka Box (Cloudeka's S3-compatible storage)

This is the counterpart to the upload notebook. **Deka Box** speaks the **S3 API**, so we use the same `boto3` client - pointed at the Deka Box endpoint - to fetch a **PDF** back out of a bucket.

In this notebook we:

1. Configure credentials and the Deka Box endpoint.
2. Build the S3 client for Deka Box.
3. List what is in the bucket.
4. Download a PDF to a local file.

A RAG pipeline uses exactly this step to pull source PDFs out of storage before ingesting them.

## Dependencies

This notebook needs `boto3` (the S3 client) and `python-dotenv` (loads credentials from a `.env` file so they stay out of the notebook). They are listed in `../requirements.txt`; install once from the `day2skk` folder:

```bash
pip install -r requirements.txt
```

## Configuration

Same setup as the upload notebook. Get your **Access Key**, **Secret Key**, and **endpoint** from the Deka Box console, then create a `.env` file next to this notebook:

```dotenv
ACCESS_KEY_ID=your-access-key
SECRET_ACCESS_KEY=your-secret-key
S3_ENDPOINT_URL=https://your-deka-box-endpoint   # from the Deka Box console
REGION=us-east-1                                 # any value works for most S3-compatible stores
S3_BUCKET=my-bucket                              # the bucket to download from
```

Never commit real keys. Keep `.env` in your `.gitignore`.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(".env")   # loads variables from a local .env file, if present

ACCESS_KEY_ID = os.environ.get("ACCESS_KEY_ID")
SECRET_ACCESS_KEY = os.environ.get("SECRET_ACCESS_KEY")
S3_ENDPOINT_URL = os.environ.get("S3_ENDPOINT_URL")
REGION = os.environ.get("REGION", "us-east-1")
S3_BUCKET = os.environ.get("S3_BUCKET")

# Sanity check without printing the secrets themselves.
print("ACCESS_KEY_ID set:    ", bool(ACCESS_KEY_ID))
print("SECRET_ACCESS_KEY set:", bool(SECRET_ACCESS_KEY))
print("S3_ENDPOINT_URL:      ", S3_ENDPOINT_URL)
print("REGION:               ", REGION)
print("S3_BUCKET:            ", S3_BUCKET)

## Build the Deka Box client

The same client as the upload notebook: a `boto3` S3 client whose `endpoint_url` points at Deka Box, with the checksum `Config` set to `when_required` for compatibility with Cloudeka/Ceph.

In [ ]:
import boto3
from botocore.config import Config


def make_client():
    """Build an S3 client pointed at Deka Box."""
    return boto3.client(
        "s3",
        region_name=REGION,
        endpoint_url=S3_ENDPOINT_URL or None,
        aws_access_key_id=ACCESS_KEY_ID,
        aws_secret_access_key=SECRET_ACCESS_KEY,
        config=Config(
            request_checksum_calculation="when_required",
            response_checksum_validation="when_required",
        ),
    )


client = make_client()
print("client ready for endpoint:", client.meta.endpoint_url)

## See what is in the bucket

Before downloading, list the objects so you know which **key** to fetch. We filter by the `uploads/` prefix used in the upload notebook; drop the prefix to see everything.

In [ ]:
listing = client.list_objects_v2(Bucket=S3_BUCKET, Prefix="uploads/")
objects = listing.get("Contents", [])

if not objects:
    print("No objects found under 'uploads/'. Run the upload notebook first, or change the prefix.")
else:
    print("Objects under 'uploads/':")
    for obj in objects:
        print(f"  {obj['Key']}  ({obj['Size']} bytes)")

## Download the PDF

We fetch the object with **`get_object`**, which returns a streaming body; `.read()` gives the raw bytes, and we write them to a local file. Reading **bytes** (not text) is exactly what a binary file like a PDF needs.

By default we download `uploads/sample.pdf` - the file created by the upload notebook. Change `object_key` to download a different PDF.

In [ ]:
from pathlib import Path
from botocore.exceptions import BotoCoreError, ClientError

object_key = "uploads/sample.pdf"           # the PDF to download
local_path = Path(Path(object_key).name)    # save under the file's base name locally


def download(client, bucket: str, key: str, dest: Path) -> None:
    print(f"Downloading s3://{bucket}/{key} -> {dest}")
    response = client.get_object(Bucket=bucket, Key=key)
    data = response["Body"].read()
    dest.write_bytes(data)
    print(f"  saved {len(data)} bytes")


try:
    download(client, S3_BUCKET, object_key, local_path)
except ClientError as exc:
    code = exc.response.get("Error", {}).get("Code")
    if code in ("NoSuchKey", "404"):
        print(f"Object not found: {object_key}. Check the listing above for the right key.")
    else:
        print("S3 error:", exc)
except BotoCoreError as exc:
    print("S3 error:", exc)

## Verify the downloaded PDF

Confirm the local file exists, check its size, and check the **PDF signature**. A PDF is binary, so instead of printing its contents we look at the first bytes: a real PDF starts with `%PDF-`.

In [ ]:
data = local_path.read_bytes()

print("exists locally:  ", local_path.exists(), "->", local_path.resolve())
print("size (bytes):    ", len(data))
print("first bytes:     ", data[:8])
print("looks like a PDF:", data.startswith(b"%PDF-"))

## Alternative: `download_file`

`boto3` also offers a one-line helper, `download_file`, which streams straight to disk (handy for large files). It is the simplest option when you just want the bytes on disk. Downloads do not hit the chunked-encoding issue that uploads do, so either approach is fine here.

In [ ]:
alt_path = Path("sample_via_download_file.pdf")
try:
    client.download_file(S3_BUCKET, object_key, str(alt_path))
    print("downloaded to", alt_path.resolve())
except (BotoCoreError, ClientError) as exc:
    print("S3 error:", exc)

## Recap

- Downloading uses the **same Deka Box client** as uploading - only the operation differs.
- **`list_objects_v2`** (optionally with a `Prefix`) shows the keys available to download.
- **`get_object(...)['Body'].read()`** returns the raw bytes, which you write to a file; **`download_file`** streams straight to disk and is convenient for large objects.
- Handle a missing key by catching `ClientError` (`NoSuchKey` / `404`).

With upload and download in place, a RAG pipeline can store its source documents in Deka Box and pull them back for ingestion.